# 01 - Read an object, then its numbers

How do I get an object as a dictionary, find the array it names, and read that array (whole or a slab) into numpy?

An object does not contain its numbers: it names arrays by a path in an HDF5 or external file. `retp` asks the server for the object (`get_object`, `get_objects`), for the array's shape (`array_metadata`) and for its values (`read_array`).

## Before you run

Settings come from environment variables only. No server name, user or token is written in this notebook, and the token is never printed.

| Variable | Needed | Meaning |
|---|---|---|
| `RETP_URL` | yes | `wss://...` ETP endpoint |
| `RETP_TOKEN` | yes | bearer token, without its scheme |
| `RETP_INSECURE` | no | `1` skips the certificate check (default: checked) |
| `RETP_PARTITION` | no | OSDU data partition |
| `RETP_DATASPACE` | no | the dataspace to read; when unset, the first one listed is taken and the notebook says so |
| `RETP_WRITE_DATASPACE` | no | a dataspace you created for the test and will delete; only the writing notebook uses it, and it does nothing without it |
| `RETP_EPC` | no | path of an EPC to push (its `.h5` beside it) |

When a precondition is missing, a cell prints why and the rest of the notebook is skipped without an error. When the server answers something the ETP 1.2 specification forbids, an `assert` fails with a message. The assertions check the **shape** of the answers, never the values a given server holds.

In [ ]:
import numpy as np
import retp
import _server

KW = _server.connect_arguments()
READY = KW is not None

In [ ]:
import re

UUID = re.compile(r"[0-9a-fA-F]{8}-(?:[0-9a-fA-F]{4}-){3}[0-9a-fA-F]{12}")
TYPE = re.compile(r"/([a-z]+[0-9]+)\.([A-Za-z0-9_]+)\(")


def uuid_of(uri):
    found = UUID.findall(uri)
    return found[-1].lower() if found else None


def family(uri):
    found = TYPE.search(uri)
    return found.group(1) if found else None


def qualified_type(uri):
    found = TYPE.search(uri)
    return f"{found.group(1)}.{found.group(2)}" if found else None


def refused(call, *expected):
    # Run call(); it must raise one of expected. Anything else is a failure.
    try:
        call()
    except expected as error:
        return error
    except Exception as other:
        raise AssertionError(f"expected {[e.__name__ for e in expected]}, got {type(other).__name__}: {other}")
    raise AssertionError("the call returned instead of refusing")

In [ ]:
session = None
DATASPACE = None
URIS = []
if READY:
    session = retp.connect(**KW)
    print(session)   # the repr names the URL and the mode, never the token
    DATASPACE = _server.dataspace_to_read(session)
    if DATASPACE is not None:
        URIS = [row["uri"] for row in session.resources(DATASPACE)["resources"]]
        print(len(URIS), "resource(s) in", DATASPACE)
        if not URIS:
            print("The dataspace holds no object: nothing more to read.")

## One object

`get_object(uri)` returns a dictionary in the EnergyML JSON encoding. The bridge documents that every node carries its `$type`; the cell checks it on every dictionary of the tree.

In [ ]:
def walk(node, path=""):
    yield path, node
    if isinstance(node, dict):
        for key, value in node.items():
            yield from walk(value, f"{path}/{key}")
    elif isinstance(node, list):
        for index, value in enumerate(node):
            yield from walk(value, f"{path}[{index}]")


if URIS:
    obj = session.get_object(URIS[0])
    assert isinstance(obj, dict), f"get_object returned a {type(obj).__name__}"
    assert isinstance(obj.get("$type"), str) and obj["$type"], "the object has no $type"
    untyped = [path for path, node in walk(obj) if isinstance(node, dict) and "$type" not in node]
    assert not untyped, f"{len(untyped)} dictionary node(s) without $type, first at {untyped[0]!r}"
    print(obj["$type"], "|", sorted(obj)[:8], "...")

    # The identifier in the object is the one in its URI. The key spelling is
    # the generated model's; look it up without assuming the case.
    identifier = next((value for key, value in obj.items() if key.lower() == "uuid"), None)
    if identifier is None:
        print("TODO(verify): no top-level key spelled 'uuid' in this object; the identity check was skipped.")
    else:
        assert str(identifier).lower() == uuid_of(URIS[0]), "the uuid in the object is not the uuid in its URI"

## Several objects, and what an error looks like

`get_objects(uris)` returns `(objects, errors)`. A URI that failed is in `errors` with `kind`, `code` and `message`; it is never silently missing. A URI that exists nowhere is built here by replacing the uuid of a real one with zeros.

TODO(verify): the keys of `objects` and `errors` are the URIs as the caller wrote them, according to the bridge's docstring; the cell asserts it.

In [ ]:
if URIS:
    batch = URIS[:5]
    last = list(UUID.finditer(URIS[0]))[-1]
    absent = URIS[0][: last.start()] + "00000000-0000-0000-0000-000000000000" + URIS[0][last.end():]
    asked = batch + [absent]

    objects, errors = session.get_objects(asked)
    assert set(objects) | set(errors) == set(asked), "an asked URI is in neither objects nor errors"
    assert not set(objects) & set(errors), "a URI is both found and failed"
    for uri, found in objects.items():
        assert isinstance(found, dict) and "$type" in found, uri
    assert absent in errors and absent not in objects, "an object that does not exist was not reported as an error"
    assert set(errors[absent]) == {"kind", "code", "message"}, sorted(errors[absent])
    print(len(objects), "found;", "the absent one:", errors[absent])

    # get_object raises where get_objects reports.
    error = refused(lambda: session.get_object(absent), retp.RetpError)
    assert not isinstance(error, ValueError)
    print("get_object(absent) ->", type(error).__name__, getattr(error, "code", None))

## Finding an array in an object

The path to ask for is in the object: RESQML 2.0 writes it as `PathInHdfFile`, RESQML 2.2 as `PathInExternalFile`. The cell walks the dictionaries of the first objects of the dataspace until one names an array, rather than hard-coding a path.

In [ ]:
PATH_KEYS = ("PathInHdfFile", "PathInExternalFile")


def array_paths(obj):
    return [
        node[key]
        for _, node in walk(obj)
        if isinstance(node, dict)
        for key in PATH_KEYS
        if isinstance(node.get(key), str) and node[key]
    ]


ARRAY = None
for start in range(0, min(len(URIS), 100), 20):
    chunk, _ = session.get_objects(URIS[start:start + 20])
    for uri, found in chunk.items():
        paths = array_paths(found)
        if paths:
            ARRAY = (uri, paths[0])
            break
    if ARRAY:
        break
if ARRAY:
    print("array", ARRAY[1], "in", qualified_type(ARRAY[0]))
elif URIS:
    print("None of the first 100 objects names an array: the array cells are skipped.")

## Its shape, then its values

`array_metadata(uri, [path])` returns `(found, errors)`. `read_array(uri, path)` returns `(values, dimensions)`: `values` is a flat numpy array in the array's logical type, row-major, and `values.reshape(dimensions)` is the caller's line. The sizes must agree with the metadata.

In [ ]:
VALUES = None
if ARRAY:
    uri, path = ARRAY
    found, errors = session.array_metadata(uri, [path])
    assert not errors, f"array_metadata failed: {errors}"
    assert len(found) == 1 and found[0]["path"] == path
    meta = found[0]
    assert set(meta) == {"path", "dimensions", "logical", "transport", "preferred_subarray_dimensions"}, sorted(meta)
    dims = list(meta["dimensions"])
    assert dims and all(isinstance(d, int) and d >= 0 for d in dims), f"dimensions {dims!r}"
    print("metadata:", dims, meta["logical"], "on the wire as", meta["transport"])

    try:
        VALUES, dimensions = session.read_array(uri, path)
    except retp.Unsupported as why:
        print("This array cannot cross as numbers:", why)
    else:
        assert isinstance(VALUES, np.ndarray) and VALUES.ndim == 1, "values must be a flat numpy array"
        assert list(dimensions) == dims, f"read_array says {list(dimensions)}, array_metadata says {dims}"
        assert VALUES.size == int(np.prod(dims)), f"{VALUES.size} values for dimensions {dims}"
        grid = VALUES.reshape(dimensions)
        assert grid.shape == tuple(dims)
        print(VALUES.dtype, grid.shape, VALUES[:5])

## A slab

`start` and `count` (one entry each per dimension) ask for a block, ETP's subarray. The block must be what the same cut of the whole array is.

In [ ]:
if VALUES is not None and VALUES.size > 0:
    count = [min(d, 2) for d in dims]
    for label, start in (
        ("first block", [0] * len(dims)),
        ("last block", [d - c for d, c in zip(dims, count)]),
    ):
        block, block_dims = session.read_array(uri, path, start=start, count=count)
        assert list(block_dims) == count, f"{label}: asked {count}, got {list(block_dims)}"
        assert block.size == int(np.prod(count)), f"{label}: {block.size} values for {count}"
        cut = tuple(slice(s, s + c) for s, c in zip(start, count))
        assert np.array_equal(block.reshape(block_dims), grid[cut], equal_nan=True), f"{label} differs from the same cut of the whole array"
        print(label, start, count, "agrees with the whole array")

## Wrong requests

`start` without `count` is a `ValueError`, and so are lists of different lengths. A block outside the array is the server's refusal: a `retp.RetpError`.

In [ ]:
if ARRAY:
    error = refused(lambda: session.read_array(uri, path, start=[0] * len(dims)), ValueError)
    assert "start" in str(error) and "count" in str(error), str(error)
    print("start without count ->", type(error).__name__)

    error = refused(lambda: session.read_array(uri, path, start=[0] * len(dims), count=[1] * (len(dims) + 1)), ValueError)
    print("start and count of different lengths ->", type(error).__name__)

    if dims and dims[0] > 0:
        too_far = [dims[0] + 1000] + [1] * (len(dims) - 1)
        error = refused(lambda: session.read_array(uri, path, start=[0] * len(dims), count=too_far), retp.RetpError)
        print("block outside the array ->", type(error).__name__, getattr(error, "code", None))
if session is not None:
    session.close()
    session.close()   # closing twice does nothing
print("done")